Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import pandas as pd

train_labels_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_labels.xlsx"
train_df = pd.read_excel(train_labels_path)
train_df.columns = train_df.columns.str.strip().str.lower()

print(train_df["label"].value_counts())
print("\n总数:", len(train_df))

label
Homophobic       705
Non_Anti_LGBT    196
Transphobic       55
Name: count, dtype: int64

总数: 956


In [4]:
import os

train_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_images"
files = os.listdir(train_image_dir)
print(f"训练集图片数量: {len(files)}")
print("前5张:", sorted(files)[:5])

训练集图片数量: 956
前5张: ['1.jpg', '10.jpg', '100.jpg', '101.gif', '102.jpg']


CLIP embedding

In [6]:
import os

for f in ["/content/drive/MyDrive/train_embeddings.npy",
          "/content/drive/MyDrive/train_meta.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"已删除: {f}")

已删除: /content/drive/MyDrive/train_embeddings.npy
已删除: /content/drive/MyDrive/train_meta.json


In [7]:
import os
import torch
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ===============================
# 1️⃣ 加载 CLIP 模型
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP loaded!")

# ===============================
# 2️⃣ 读取训练集标签
# ===============================
train_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_images"
train_labels_path = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_labels.xlsx"

train_df = pd.read_excel(train_labels_path)
train_df.columns = train_df.columns.str.strip().str.lower()
train_df["image_id"] = train_df["id"].astype(str).str.strip()

# ===============================
# 3️⃣ 生成训练集 embedding
# ===============================
embeddings = []
valid_ids = []
valid_labels = []
valid_filenames = []

all_files = os.listdir(train_image_dir)
id_to_file = {}
for f in all_files:
    fid = os.path.splitext(f)[0]
    id_to_file[fid] = f

print("生成训练集 embeddings...")
for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    img_id = str(row["image_id"])
    label = row["label"]

    if img_id not in id_to_file:
        continue

    img_path = os.path.join(train_image_dir, id_to_file[img_id])
    try:
        image = Image.open(img_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt")
        with torch.no_grad():
            # 关键修正：直接用 vision_model 取 pooler_output
            outputs = clip_model.vision_model(**inputs)
            emb = outputs.pooler_output  # shape: [1, 768]
            emb = emb / emb.norm(dim=-1, keepdim=True)
        embeddings.append(emb.squeeze().numpy())
        valid_ids.append(img_id)
        valid_labels.append(label)
        valid_filenames.append(id_to_file[img_id])
    except Exception as e:
        print(f"跳过 {img_id}: {e}")

embeddings = np.array(embeddings)
print(f"完成！共 {len(embeddings)} 个训练集 embeddings")

np.save("/content/drive/MyDrive/train_embeddings.npy", embeddings)
meta = {"ids": valid_ids, "labels": valid_labels, "filenames": valid_filenames}
with open("/content/drive/MyDrive/train_meta.json", "w") as f:
    json.dump(meta, f)
print("Embeddings 已保存！")

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded!
生成训练集 embeddings...


  9%|▊         | 83/956 [00:28<02:31,  5.78it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 956/956 [03:17<00:00,  4.84it/s]

完成！共 956 个训练集 embeddings
Embeddings 已保存！


跑批量推理代码，对每张测试图片动态检索最相似的 3 个训练样本作为 few-shot 示例

In [9]:
import json

with open("/content/drive/MyDrive/GPT4omini_HM_FewShot_RAG_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

unknowns = [p for p in predictions if p["predicted_label"] == "UNKNOWN"]
print(f"UNKNOWN 数量: {len(unknowns)}")
for u in unknowns[:3]:
    print(f"\n图片: {u['image_name']}")
    print(f"原始输出: {u['raw_output']}")

UNKNOWN 数量: 3

图片: 3.jpg
原始输出: I'm unable to analyze and classify the content as requested.

图片: 6.jpg
原始输出: I'm unable to classify or analyze these memes. If you have any other questions or need assistance with a different topic, feel free to ask!

图片: 8.jpg
原始输出: I'm unable to classify harmful content directly from the provided images. However, if you describe the content or text appearing in the memes, I can help analyze for harmfulness based on your description.


In [10]:
import os

f = "/content/drive/MyDrive/GPT4omini_HM_FewShot_RAG_pred.json"
if os.path.exists(f):
    os.remove(f)
    print("已删除")

已删除


In [12]:
import os
import json
import base64
import re
import time
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from openai import OpenAI
import torch

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
train_image_dir = "/content/drive/MyDrive/MyThesis2026/Chinese/Train/Train_images"
output_json = "/content/drive/MyDrive/GPT4omini_HM_FewShot_RAG_pred.json"

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/train_embeddings.npy")
with open("/content/drive/MyDrive/train_meta.json", "r") as f:
    train_meta = json.load(f)

train_ids = train_meta["ids"]
train_labels = train_meta["labels"]
train_filenames = train_meta["filenames"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in ["Homophobic", "Transphobic", "Non_Anti_LGBT"]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

# ===============================
# 构造 Few-shot Prompt（纯文字示例）
# ===============================
def build_prompt(example_indices):
    label_map = {
        "Homophobic": "Homophobia",
        "Transphobic": "Transphobia",
        "Non_Anti_LGBT": "Non_LGBT"
    }

    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    prompt = f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is harmful or not.

Below are 3 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing the image and text, using the provided examples as reference to determine its harmfulness.
Step 2: If the meme contains any negative or insulting reference to gay or lesbian people, output Homophobia.
Step 3: If the meme contains any negative or insulting reference to transgender people, output Transphobia.
Step 4: If neither of the above applies, output Non_LGBT.

Output:
Your output should strictly follow the format:
Class labels: Homophobia, Transphobia, or Non_LGBT
Thought: Give your reason here"""

    return prompt

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# ===============================
# API 调用（带重试）
# ===============================
def call_with_retry(content, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": content}],
                max_tokens=200
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)
                print(f"  Rate limit，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

# ===============================
# 获取测试图片列表
# ===============================
image_files = sorted(
    [f for f in os.listdir(test_image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif"))],
    key=lambda x: int(re.search(r"(\d+)", x).group(1)) if re.search(r"(\d+)", x) else 0
)

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] not in ["ERROR"]]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining = [f for f in image_files if f not in done_images]
print(f"剩余待处理: {len(remaining)} 张")

# ===============================
# 批量推理
# ===============================
for img_name in tqdm(remaining, desc="推理进度"):
    img_path = os.path.join(test_image_dir, img_name)

    try:
        # 1. 获取测试图片 embedding
        test_emb = get_embedding(img_path)

        # 2. RAG 检索
        example_indices = retrieve_examples(test_emb)

        # 3. 构造 prompt
        prompt_text = build_prompt(example_indices)

        # 4. 构造 content（只传测试图片，示例用文字）
        ext = img_name.lower().split(".")[-1]
        mime = "image/png" if ext == "png" else "image/jpeg"
        test_img_data = encode_image(img_path)

        content = [
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{test_img_data}"}},
            {"type": "text", "text": prompt_text}
        ]

        # 5. 调用 API
        raw = call_with_retry(content)

        # 6. 解析 label
        m = re.search(r"(Homophobia|Transphobia|Non_LGBT)", raw, re.IGNORECASE)
        if m:
            label = m.group(1)
        elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
            label = "Non_LGBT"
        else:
            label = "UNKNOWN"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw
        })

        print(f"✅ {img_name} -> {label}")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(5)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e)
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(5)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


训练集 embeddings 加载完成，共 956 条
没有已有结果，从头开始...
剩余待处理: 232 张


推理进度:   0%|          | 0/232 [00:00<?, ?it/s]

...

✅ 238.jpg -> Non_LGBT


推理进度: 100%|█████████▉| 231/232 [30:07<00:07,  7.79s/it]

✅ 239.jpg -> Non_LGBT


推理进度: 100%|██████████| 232/232 [30:15<00:00,  7.83s/it]


完成！共 232 条结果已保存
  Homophobia: 31
  Non_LGBT: 198
  Transphobia: 3
